
# Financial Performance Analysis Project

## Objetivo
En este proyecto analizamos el desempeño financiero de una empresa a partir de datos mensuales de ingresos, costos, gastos y presupuesto.

Este notebook está pensado para:
- practicar análisis financiero aplicado
- construir un proyecto de portfolio para CV y GitHub
- aprender conceptos básicos como **profit**, **margin**, **budget vs actual** y **variance analysis**

---

## Dataset
Variables incluidas:
- `date`: mes
- `revenue`: ingresos
- `cost`: costos
- `expenses`: gastos operativos
- `budget_revenue`: presupuesto de ingresos

El dataset incluye errores intencionales para simular un caso real:
- un valor nulo
- una fecha mal formateada
- un outlier en costos


## 1. Importación de librerías

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')


## 2. Carga de datos

In [ ]:

df = pd.read_csv("financials_dataset.csv")
df.head()



### Qué revisar al cargar el archivo
Antes de analizar, siempre conviene validar:
- cantidad de filas y columnas
- tipos de datos
- valores nulos
- formatos inconsistentes


In [ ]:

print("Dimensiones:", df.shape)
print("\nTipos de datos:")
print(df.dtypes)

print("\nValores nulos:")
print(df.isnull().sum())


## 3. Exploración inicial

In [ ]:

df.describe(include='all')



## 4. Calidad de datos

En esta sección vamos a detectar tres problemas típicos:
1. fechas inconsistentes
2. valores faltantes
3. outliers


In [ ]:

# Revisamos las fechas originales
df['date']



### 4.1 Conversión de fechas
Como una fila tiene formato diferente, usamos `errors='coerce'` para detectar problemas.


In [ ]:

df['date_parsed'] = pd.to_datetime(df['date'], errors='coerce')
df[['date', 'date_parsed']].head(20)



Como una fecha quedó fuera del formato esperado, la vamos a volver a parsear con una segunda estrategia.


In [ ]:

mask_invalid = df['date_parsed'].isna()
df.loc[mask_invalid, 'date_parsed'] = pd.to_datetime(df.loc[mask_invalid, 'date'], dayfirst=True, errors='coerce')
df[['date', 'date_parsed']].tail(10)



### 4.2 Nulos
Tenemos un valor nulo en `revenue`. Una estrategia simple y defendible para series mensuales es imputar con interpolación.


In [ ]:

df['revenue']


In [ ]:

df = df.sort_values('date_parsed').reset_index(drop=True)
df['revenue'] = df['revenue'].interpolate(method='linear')
df['revenue']



### 4.3 Detección de outliers
Vamos a usar IQR sobre la columna `cost`.


In [ ]:

Q1 = df['cost'].quantile(0.25)
Q3 = df['cost'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df['cost'] < lower_bound) | (df['cost'] > upper_bound)]
outliers



Como es un proyecto de portfolio, podemos explicar dos enfoques:
- **mantener el outlier** si refleja un evento real
- **caparlo o reemplazarlo** si parece error de carga

Acá vamos a winsorizar el valor al límite superior para evitar distorsión excesiva.


In [ ]:

df['cost_clean'] = df['cost'].clip(upper=upper_bound)
df[['date', 'cost', 'cost_clean']][df['cost'] != df['cost_clean']]


## 5. Dataset limpio

In [ ]:

df_clean = df.copy()

df_clean['date'] = df_clean['date_parsed']
df_clean['cost'] = df_clean['cost_clean']

df_clean = df_clean.drop(columns=['date_parsed', 'cost_clean'])
df_clean = df_clean.sort_values('date').reset_index(drop=True)

df_clean.head()



## 6. Feature engineering

Creamos métricas financieras clave:
- `profit`
- `margin`
- `variance`
- `variance_pct`


In [ ]:

df_clean['profit'] = df_clean['revenue'] - df_clean['cost'] - df_clean['expenses']
df_clean['margin'] = df_clean['profit'] / df_clean['revenue']
df_clean['variance'] = df_clean['revenue'] - df_clean['budget_revenue']
df_clean['variance_pct'] = df_clean['variance'] / df_clean['budget_revenue']

df_clean.head()


## 7. KPIs principales

In [ ]:

total_revenue = df_clean['revenue'].sum()
total_profit = df_clean['profit'].sum()
avg_margin = df_clean['margin'].mean()
total_variance = df_clean['variance'].sum()

print(f"Total Revenue: {total_revenue:,.2f}")
print(f"Total Profit: {total_profit:,.2f}")
print(f"Average Margin: {avg_margin:.2%}")
print(f"Total Variance vs Budget: {total_variance:,.2f}")



## 8. Análisis temporal

### 8.1 Revenue vs Budget


In [ ]:

plt.figure(figsize=(12,5))
plt.plot(df_clean['date'], df_clean['revenue'], marker='o', label='Revenue')
plt.plot(df_clean['date'], df_clean['budget_revenue'], marker='o', label='Budget Revenue')
plt.title('Revenue vs Budget')
plt.xlabel('Date')
plt.ylabel('Amount')
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()


### 8.2 Profit mensual

In [ ]:

plt.figure(figsize=(12,5))
plt.plot(df_clean['date'], df_clean['profit'], marker='o')
plt.title('Monthly Profit')
plt.xlabel('Date')
plt.ylabel('Profit')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


### 8.3 Variance mensual

In [ ]:

plt.figure(figsize=(12,5))
plt.bar(df_clean['date'].dt.strftime('%Y-%m'), df_clean['variance'])
plt.title('Monthly Variance vs Budget')
plt.xlabel('Date')
plt.ylabel('Variance')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


### 8.4 Margen mensual

In [ ]:

plt.figure(figsize=(12,5))
plt.plot(df_clean['date'], df_clean['margin'], marker='o')
plt.title('Monthly Profit Margin')
plt.xlabel('Date')
plt.ylabel('Margin')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()



## 9. Meses con peor desempeño

Acá detectamos:
- menores ganancias
- peores variaciones contra presupuesto


In [ ]:

worst_profit_months = df_clean.nsmallest(5, 'profit')[['date', 'revenue', 'cost', 'expenses', 'profit']]
worst_variance_months = df_clean.nsmallest(5, 'variance')[['date', 'revenue', 'budget_revenue', 'variance']]

print("Peores meses por profit")
display(worst_profit_months)

print("Peores meses por variance")
display(worst_variance_months)



## 10. Forecast simple de revenue

No buscamos precisión de modelo avanzada, sino mostrar criterio analítico.
Usamos una regresión lineal sencilla contra el tiempo.


In [ ]:

from sklearn.linear_model import LinearRegression

forecast_df = df_clean[['date', 'revenue']].copy()
forecast_df['time_index'] = np.arange(len(forecast_df))

X = forecast_df[['time_index']]
y = forecast_df['revenue']

model = LinearRegression()
model.fit(X, y)

future_index = np.arange(len(forecast_df), len(forecast_df)+3)
future_pred = model.predict(future_index.reshape(-1, 1))

last_date = forecast_df['date'].max()
future_dates = pd.date_range(last_date + pd.offsets.MonthEnd(1), periods=3, freq='ME')

forecast_result = pd.DataFrame({
    'date': future_dates,
    'forecast_revenue': future_pred
})

forecast_result


In [ ]:

plt.figure(figsize=(12,5))
plt.plot(df_clean['date'], df_clean['revenue'], marker='o', label='Historical Revenue')
plt.plot(forecast_result['date'], forecast_result['forecast_revenue'], marker='o', linestyle='--', label='Forecast Revenue')
plt.title('Revenue Forecast - Next 3 Months')
plt.xlabel('Date')
plt.ylabel('Revenue')
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()



## 11. Insights de negocio

Ejemplo de conclusiones que podrías escribir en GitHub o contar en entrevista:

1. La empresa presenta meses con ingresos por debajo del presupuesto, lo que evidencia desvíos negativos en ciertos períodos.
2. La rentabilidad se ve afectada tanto por la relación costo/ingreso como por el peso de los gastos operativos.
3. Un valor atípico en costos puede distorsionar el análisis si no se trata adecuadamente.
4. El seguimiento de `variance` y `margin` es clave para la toma de decisiones financieras.
5. Un forecast simple permite proyectar ingresos futuros y apoyar procesos de planning.

---

## 12. Próximos pasos para mejorar el proyecto

Para llevar este proyecto a un nivel más fuerte de portfolio:
- agregar segmentación por unidad de negocio o región
- incorporar análisis trimestral
- construir dashboard en Power BI
- comparar actual vs budget vs forecast
- almacenar los datos en SQL y consultar desde Python



## 13. Export opcional del dataset limpio


In [ ]:

df_clean.to_csv("financials_clean.csv", index=False)
forecast_result.to_csv("financials_forecast.csv", index=False)
print("Archivos exportados correctamente.")
